In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [2]:
def get_schedule(year):
    url = f"https://www.espn.com/nba/team/schedule/_/name/cha/season/{year}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to fetch schedule for {year}")
            return []
        
        soup = BeautifulSoup(response.content, 'html.parser')
        # Find all game links
        # Links look like /nba/game/_/gameId/401584698
        links = soup.find_all('a', href=re.compile(r'/nba/game/_/gameId/\d+'))
        game_ids = set()
        for link in links:
            match = re.search(r'gameId/(\d+)', link['href'])
            if match:
                game_ids.add(match.group(1))
        
        return list(game_ids)
    except Exception as e:
        print(f"Error getting schedule for {year}: {e}")
        return []

# Test
ids_2024 = get_schedule(2024)
print(f"Found {len(ids_2024)} games for 2024")

Found 82 games for 2024


In [3]:
ARENA_CAPACITIES = {
    # Current NBA Arenas (as of 2023-24)
    "State Farm Arena": 16600,       # Atlanta Hawks
    "TD Garden": 19156,              # Boston Celtics
    "Barclays Center": 17732,        # Brooklyn Nets
    "Spectrum Center": 19077,        # Charlotte Hornets
    "United Center": 20917,          # Chicago Bulls
    "Rocket Mortgage FieldHouse": 19432, # Cleveland Cavaliers
    "American Airlines Center": 19200, # Dallas Mavericks
    "Ball Arena": 19605,             # Denver Nuggets
    "Little Caesars Arena": 20332,   # Detroit Pistons
    "Chase Center": 18064,           # Golden State Warriors
    "Toyota Center": 18055,          # Houston Rockets
    "Gainbridge Fieldhouse": 17923,  # Indiana Pacers
    "Crypto.com Arena": 19068,       # LA Clippers / Lakers
    "FedExForum": 17794,             # Memphis Grizzlies
    "Kaseya Center": 19600,          # Miami Heat
    "Fiserv Forum": 17341,           # Milwaukee Bucks
    "Target Center": 18024,          # Minnesota Timberwolves
    "Smoothie King Center": 16867,   # New Orleans Pelicans
    "Madison Square Garden": 19812,  # New York Knicks
    "Paycom Center": 18203,          # Oklahoma City Thunder
    "Kia Center": 18846,             # Orlando Magic
    "Wells Fargo Center": 20478,     # Philadelphia 76ers
    "Footprint Center": 17071,       # Phoenix Suns
    "Moda Center": 19441,            # Portland Trail Blazers
    "Golden 1 Center": 17608,        # Sacramento Kings
    "Frost Bank Center": 18418,      # San Antonio Spurs
    "Scotiabank Arena": 19800,       # Toronto Raptors
    "Delta Center": 18206,           # Utah Jazz
    "Capital One Arena": 20356,      # Washington Wizards
    
    # Older names/Previous Arenas
    "Staples Center": 19068,         # Old name for Crypto.com
    "Pepsi Center": 19605,           # Old name for Ball Arena
    "Quicken Loans Arena": 19432,    # Old name for Rocket Mortgage
    "Gund Arena": 19432,             # Older name for Rocket Mortgage
    "Philips Arena": 16600,          # Old name for State Farm
    "Time Warner Cable Arena": 19077,# Old name for Spectrum Center
    "Charlotte Bobcats Arena": 19077,# Old name for Spectrum Center
    "Amway Center": 18846,           # Old name for Kia Center
    "AT&T Center": 18418,            # Old name for Frost Bank Center
    "Talking Stick Resort Arena": 17071, # Old name for Footprint Center
    "Phoenix Suns Arena": 17071,     # Old name for Footprint Center
    "Bankers Life Fieldhouse": 17923,# Old name for Gainbridge
    "Chesapeake Energy Arena": 18203,# Old name for Paycom
    "Vivint Arena": 18206,           # Old name for Delta Center
    "Vivint Smart Home Arena": 18206,# Old name for Delta Center
    "EnergySolutions Arena": 18206,  # Old name for Delta Center
    "AmericanAirlines Arena": 19600, # Old name for Kaseya Center
    "Air Canada Centre": 19800,      # Old name for Scotiabank Arena
    "Verizon Center": 20356,         # Old name for Capital One
    "MCI Center": 20356,             # Older name for Capital One
    "Xfinity Mobile Arena": 20478,     # Old name for Wells Fargo Center
    "crypto.com Arena": 20000,       # Old name for Crypto.com Arena
    "Mortgage Matchup Center": 18422, # Temporary name for Rocket Mortgage FieldHouse during 2023-24 season
    "Toyota Center (Houston)": 18500, # To avoid confusion with Toyota Center in Kennewick, WA (used for some preseason games)
    "Intuit Dome": 18300,             # Future home of the LA Clippers (opening in 2024-25)
    
    # Historical Arenas requested
    "The Palace of Auburn Hills": 22076,
    "Oracle Arena": 19596,
    "Sleep Train Arena": 17317,
    "BMO Harris Bradley Center": 18717,
    "Bercy Arena": 15609,            # NBA Paris Games
    "Accor Arena": 15609,            # Modern name for Bercy
    "Benchmark International Arena": 20500, # 2020-21 Raptors temporary home (Amalie Arena)
}

In [4]:
from datetime import datetime

def get_game_data(game_id):
    # Use internal API for cleaner data
    url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return None
            
        data = response.json()
        
        # Initialize
        game_result = {
            'game_id': game_id,
            'date': None,
            'opponent': None,
            'city': None,
            'state': None, 
            'arena': None,
            'attendance': None,
            'capacity': None,
            'attendance_percent': None,
            'hornets_record': None,
            'opponent_record': None,
            'hornets_score': None,
            'opponent_score': None,
            'location': None,
            # Leaders
            'hornets_top_scorer (points)': None,
            'opponents_top_scorer': None,
            'hornets_most_assists': None,
            'opponents_most_assists': None,
            'hornets_top_rebounder': None,
            'opponents_top_rebounder': None
        }
        
        # Game Info
        if 'gameInfo' in data:
            info = data['gameInfo']
            att = info.get('attendance')
            game_result['attendance'] = att
            
            if 'venue' in info:
                venue = info['venue']
                arena_name = venue.get('fullName')
                game_result['arena'] = arena_name
                if 'address' in venue:
                    game_result['city'] = venue['address'].get('city')
                    game_result['state'] = venue['address'].get('state')
                
                # Capacity Logic
                if arena_name in ARENA_CAPACITIES:
                    cap = ARENA_CAPACITIES[arena_name]
                    game_result['capacity'] = cap
                    if att and cap > 0:
                        pct = (att / cap) * 100
                        game_result['attendance_percent'] = f"{pct:.1f}%"
        
        # Teams and Records
        if 'header' in data and 'competitions' in data['header']:
            competition = data['header']['competitions'][0]
            
            # Format Date
            date_str = competition.get('date')
            if date_str:
                try:
                    dt_obj = datetime.strptime(date_str, "%Y-%m-%dT%H:%MZ")
                    game_result['date'] = dt_obj.strftime("%m/%d/%Y")
                except ValueError:
                    game_result['date'] = date_str[:10]
            
            for competitor in competition.get('competitors', []):
                team = competitor.get('team', {})
                name = team.get('displayName', '')
                
                # Format Record
                record_raw = competitor.get('record', [{}])[0].get('summary', 'N/A') if competitor.get('record') else 'N/A'
                record = f"'{record_raw}" if record_raw != 'N/A' else record_raw
                
                score = competitor.get('score')
                
                if 'Hornets' in name:
                    game_result['hornets_record'] = record
                    game_result['hornets_score'] = score
                    home_away_status = competitor.get('homeAway', '')
                    game_result['location'] = 'Home' if home_away_status == 'home' else 'Away'
                else:
                    game_result['opponent'] = name
                    game_result['opponent_record'] = record
                    game_result['opponent_score'] = score

        # Leaders
        if 'leaders' in data:
            for team_leaders in data['leaders']:
                team_name = team_leaders.get('team', {}).get('displayName', '')
                is_hornets = 'Hornets' in team_name
                
                for category in team_leaders.get('leaders', []):
                    cat_name = category.get('name', '')
                    leader_list = category.get('leaders', [])
                    
                    if leader_list:
                        leader = leader_list[0]
                        athlete = leader.get('athlete', {})
                        player_name = athlete.get('displayName', '')
                        value = leader.get('displayValue', '')
                        
                        entry_val = f"{player_name} ({value})" 
                        
                        if is_hornets:
                            if cat_name == 'points':
                                game_result['hornets_top_scorer (points)'] = entry_val
                            elif cat_name == 'assists':
                                game_result['hornets_most_assists'] = entry_val
                            elif cat_name == 'rebounds':
                                game_result['hornets_top_rebounder'] = entry_val
                        else:
                            if cat_name == 'points':
                                game_result['opponents_top_scorer'] = entry_val
                            elif cat_name == 'assists':
                                game_result['opponents_most_assists'] = entry_val
                            elif cat_name == 'rebounds':
                                game_result['opponents_top_rebounder'] = entry_val

        return game_result

    except Exception as e:
        print(f"Error fetching {game_id}: {e}")
        return None

In [ ]:
from tqdm import tqdm
import os

all_games_data = []
years = range(2015, 2026)

for year in years:
    print(f"Fetching schedule for {year}...")
    game_ids = get_schedule(year)
    print(f"Found {len(game_ids)} games for {year}")
    
    for game_id in tqdm(game_ids, desc=f"Games {year}"):
        data = get_game_data(game_id)
        if data:
            data['season'] = year
            all_games_data.append(data)
        time.sleep(0.05)
            
    # Save checkpoint after each year
    pd.DataFrame(all_games_data).to_csv('hornets_games_partial.csv', index=False)

# Final Save to local repository
output_path = 'hornets_games_2015-2026.csv'
df = pd.DataFrame(all_games_data)
df.to_csv(output_path, index=False)
print(f"Done! Saved to {output_path}")
print(df.head())
print(df.columns)

Fetching schedule for 2015...
Found 82 games for 2015


Games 2015: 100%|██████████| 82/82 [01:01<00:00,  1.34it/s]


Fetching schedule for 2016...
Found 82 games for 2016


Games 2016: 100%|██████████| 82/82 [01:01<00:00,  1.33it/s]


Fetching schedule for 2017...
Found 82 games for 2017


...

Fetching schedule for 2024...
Found 82 games for 2024


Games 2024: 100%|██████████| 82/82 [00:52<00:00,  1.56it/s]


Fetching schedule for 2025...
Found 82 games for 2025


Games 2025: 100%|██████████| 82/82 [00:52<00:00,  1.56it/s]

Done! Saved to C:\Users\Matt Shaver\Downloads\hornets_games_2015-2026.csv
     game_id        date           opponent            city state  \
0  400579343  03/24/2015      Chicago Bulls         Chicago    IL   
1  400578981  01/29/2015  San Antonio Spurs     San Antonio    TX   
2  400579137  02/26/2015      Chicago Bulls         Chicago    IL   
3  400578490  11/25/2014        LA Clippers       Charlotte    NC   
4  400579289  03/17/2015          Utah Jazz  Salt Lake City    UT   

               arena  attendance  capacity attendance_percent hornets_record  \
0      United Center       21646   20917.0             103.5%         '30-39   
1  Frost Bank Center       18581   18418.0             100.9%         '19-27   
2      United Center       21509   20917.0             102.8%         '23-32   
3    Spectrum Center       17180   19077.0              90.1%          '4-11   
4       Delta Center       16743   18206.0              92.0%         '29-36   

   ... hornets_score opponent_

In [ ]:
# Commit and Push to GitHub
import subprocess

def git_commit_push():
    try:
        subprocess.run(["git", "add", "hornets_games_2015-2026.csv"], check=True)
        subprocess.run(["git", "commit", "-m", "Updated Hornets game schedule data"], check=True)
        subprocess.run(["git", "push"], check=True)
        print("Successfully pushed changes to GitHub.")
    except subprocess.CalledProcessError as e:
        print(f"Error during git operation: {e}")

# Uncomment the line below to run the commit and push automatically
# git_commit_push()